# How a GPU Runs Deep Learning: Memory, the Roofline, and a First Kernel

The full code of [How a GPU Runs Deep Learning: Memory, the Roofline, and a First Kernel](https://residual-stream.lvtan71.workers.dev/posts/how-a-gpu-runs-deep-learning/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/how-a-gpu-runs-deep-learning) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.



**Hardware:** Needs an Apple-silicon Mac: the kernels are written in Metal, which Colab can't run. Run it locally with Jupyter (see the repo's README); the cells that are only arithmetic run anywhere. About 2 min end to end on the post's Apple M4.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on Apple M4 on 2026-10-18** (torch 2.11.0, 1 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.11.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "how-a-gpu-runs-deep-learning"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/how-a-gpu-runs-deep-learning


## `bandwidth.py`

Memory bandwidth from the vector add and PyTorch's copy, and how a strided copy loses it (coalescing).

In [2]:
!{sys.executable} bandwidth.py

/Users/tanle/Developer/residual-stream-code/work/ce/venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


vector_add: read 2 arrays, write 1, in 9.8 ms = 82 GB/s


PyTorch copy: 78 GB/s


strided_copy, stride    1:    83 GB/s


strided_copy, stride    3:    45 GB/s


strided_copy, stride    9:    17 GB/s


strided_copy, stride   33:    10 GB/s


strided_copy, stride 1025:    10 GB/s


The post printed:

```text
vector_add: read 2 arrays, write 1, in 8.8 ms = 91 GB/s
PyTorch copy: 92 GB/s
strided_copy, stride    1:    93 GB/s
strided_copy, stride    3:    48 GB/s
strided_copy, stride    9:    17 GB/s
strided_copy, stride   33:    10 GB/s
strided_copy, stride 1025:     9 GB/s
```

## `roofline.py`

Vector add, matrix-vector and matrix multiplies of growing size, timed: intensity, GFLOP/s and GB/s on the roofline.

In [3]:
!{sys.executable} roofline.py

/Users/tanle/Developer/residual-stream-code/work/ce/venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


vector add             intensity     0.08 FLOP/byte       7.9 GFLOP/s     94.2 GB/s
matrix-vector, n=4096  intensity     0.50 FLOP/byte      46.0 GFLOP/s     92.0 GB/s
matrix-vector, n=8192  intensity     0.50 FLOP/byte      46.2 GFLOP/s     92.5 GB/s
matmul, n=64           intensity    10.67 FLOP/byte      38.3 GFLOP/s      3.6 GB/s
matmul, n=128          intensity    21.33 FLOP/byte     293.8 GFLOP/s     13.8 GB/s
matmul, n=256          intensity    42.67 FLOP/byte    1884.4 GFLOP/s     44.2 GB/s
matmul, n=512          intensity    85.33 FLOP/byte    3018.3 GFLOP/s     35.4 GB/s
matmul, n=1024         intensity   170.67 FLOP/byte    3351.0 GFLOP/s     19.6 GB/s
matmul, n=2048         intensity   341.33 FLOP/byte    3435.7 GFLOP/s     10.1 GB/s
matmul, n=4096         intensity   682.67 FLOP/byte    3429.4 GFLOP/s      5.0 GB/s
1,024 matmuls n=64, bmm intensity    10.67 FLOP/byte     981.5 GFLOP/s     92.0 GB/s


The post printed:

```text
vector add             intensity     0.08 FLOP/byte       6.1 GFLOP/s     73.2 GB/s
matrix-vector, n=4096  intensity     0.50 FLOP/byte      39.6 GFLOP/s     79.2 GB/s
matrix-vector, n=8192  intensity     0.50 FLOP/byte      39.7 GFLOP/s     79.5 GB/s
matmul, n=64           intensity    10.67 FLOP/byte      25.3 GFLOP/s      2.4 GB/s
matmul, n=128          intensity    21.33 FLOP/byte     273.8 GFLOP/s     12.8 GB/s
matmul, n=256          intensity    42.67 FLOP/byte    1325.1 GFLOP/s     31.1 GB/s
matmul, n=512          intensity    85.33 FLOP/byte    2778.1 GFLOP/s     32.6 GB/s
matmul, n=1024         intensity   170.67 FLOP/byte    2749.2 GFLOP/s     16.1 GB/s
matmul, n=2048         intensity   341.33 FLOP/byte    2877.5 GFLOP/s      8.4 GB/s
matmul, n=4096         intensity   682.67 FLOP/byte    2662.8 GFLOP/s      3.9 GB/s
1,024 matmuls n=64, bmm intensity    10.67 FLOP/byte     728.5 GFLOP/s     68.3 GB/s
```

## `matmul_ladder.py`

The four hand-written matmul kernels against PyTorch's, at n = 512, 1024, 2048, each checked against a float64 reference.

In [4]:
!{sys.executable} matmul_ladder.py

/Users/tanle/Developer/residual-stream-code/work/ce/venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


n =  512:   naive   309 GFLOP/s   tiled   581 GFLOP/s   tiled 32   639 GFLOP/s   registers  1334 GFLOP/s   PyTorch  2990 GFLOP/s   (all within 9e-07 of float64)


n = 1024:   naive   333 GFLOP/s   tiled   596 GFLOP/s   tiled 32   660 GFLOP/s   registers  1509 GFLOP/s   PyTorch  3340 GFLOP/s   (all within 1e-06 of float64)


n = 2048:   naive   338 GFLOP/s   tiled   590 GFLOP/s   tiled 32   629 GFLOP/s   registers  1459 GFLOP/s   PyTorch  3219 GFLOP/s   (all within 2e-06 of float64)


The post printed:

```text
n =  512:   naive   237 GFLOP/s   tiled   518 GFLOP/s   tiled 32   561 GFLOP/s   registers  1325 GFLOP/s   PyTorch  2869 GFLOP/s   (all within 1e-06 of float64)
n = 1024:   naive   291 GFLOP/s   tiled   514 GFLOP/s   tiled 32   581 GFLOP/s   registers  1308 GFLOP/s   PyTorch  3081 GFLOP/s   (all within 1e-06 of float64)
n = 2048:   naive   269 GFLOP/s   tiled   491 GFLOP/s   tiled 32   517 GFLOP/s   registers  1201 GFLOP/s   PyTorch  2867 GFLOP/s   (all within 2e-06 of float64)
```

## `tile_intensity.py`

Arithmetic: reads per output and intensity for each tile size, and the two ridge points.

In [5]:
!{sys.executable} tile_intensity.py

ridge points: this GPU 31 FLOP/byte, H100 295 FLOP/byte

   C tile  reads per output     intensity   threads, one output each
    naive        2.000 x n     0.25 FLOP/B                  1
  16 x 16        0.125 x n     4.00 FLOP/B                256
  32 x 32        0.062 x n     8.00 FLOP/B              1,024
  64 x 64        0.031 x n    16.00 FLOP/B              4,096
128 x 128        0.016 x n    32.00 FLOP/B             16,384


The post printed:

```text
ridge points: this GPU 31 FLOP/byte, H100 295 FLOP/byte

   C tile  reads per output     intensity   threads, one output each
    naive        2.000 x n     0.25 FLOP/B                  1
  16 x 16        0.125 x n     4.00 FLOP/B                256
  32 x 32        0.062 x n     8.00 FLOP/B              1,024
  64 x 64        0.031 x n    16.00 FLOP/B              4,096
128 x 128        0.016 x n    32.00 FLOP/B             16,384
```

## `smem_probe.py`

Compiles a probe kernel with 16, 32 and 64 KB of threadgroup memory to find this GPU's limit.

In [6]:
!{sys.executable} smem_probe.py

/Users/tanle/Developer/residual-stream-code/work/ce/venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


16 KB of threadgroup memory: ok
32 KB of threadgroup memory: ok


64 KB of threadgroup memory: refused (Threadgroup memory size (65536) exceeds the maximum threadgr)


The post printed:

```text
16 KB of threadgroup memory: ok
32 KB of threadgroup memory: ok
64 KB of threadgroup memory: refused (Threadgroup memory size (65536) exceeds the maximum threadgr)
```

## `waves.py`

Arithmetic: 128 x 128 tiles in waves over an H100's 132 SMs, and how much of each wave is useful.

In [7]:
!{sys.executable} waves.py

n = 1024:   64 tiles ->  1 waves; SMs busy  48%, useful work in the tiles 100%, overall  48%
n = 1408:  121 tiles ->  1 waves; SMs busy  92%, useful work in the tiles 100%, overall  92%
n = 1536:  144 tiles ->  2 waves; SMs busy  55%, useful work in the tiles 100%, overall  55%
n = 4096: 1024 tiles ->  8 waves; SMs busy  97%, useful work in the tiles 100%, overall  97%
n = 4097: 1089 tiles ->  9 waves; SMs busy  92%, useful work in the tiles  94%, overall  86%
n = 4224: 1089 tiles ->  9 waves; SMs busy  92%, useful work in the tiles 100%, overall  92%


The post printed:

```text
n = 1024:   64 tiles ->  1 waves; SMs busy  48%, useful work in the tiles 100%, overall  48%
n = 1408:  121 tiles ->  1 waves; SMs busy  92%, useful work in the tiles 100%, overall  92%
n = 1536:  144 tiles ->  2 waves; SMs busy  55%, useful work in the tiles 100%, overall  55%
n = 4096: 1024 tiles ->  8 waves; SMs busy  97%, useful work in the tiles 100%, overall  97%
n = 4097: 1089 tiles ->  9 waves; SMs busy  92%, useful work in the tiles  94%, overall  86%
n = 4224: 1089 tiles ->  9 waves; SMs busy  92%, useful work in the tiles 100%, overall  92%
```